# 12 - Task 3: features built from raw text, and the baseline re-run

**The premise.** Three rounds have now optimised the *model* while holding the feature
representation fixed, and all three ran out of room: round 3's ensemble gained 0.0017
locally and nothing on Kaggle, round 4's best combiner gained 0.0075 out-of-fold AUC and
0.0017 on Kaggle. The representation is the part nobody has touched.

**Why the supplied features are the wrong basis for this particular test set.** They are
top-5000 TF-IDF over lemmas with stop words removed, which is a pure *content* signal.
The COLING paper shows the test split is drawn from CUDRT, IELTS, NLPeer, PeerSum and
MixSet, none of which appear in training, so the content vocabulary of the test set is
substantially not the vocabulary the features were built on. Function words, punctuation,
orthography and layout do transfer across topics, and the course preprocessing removed
precisely those.

`src/text_features.py` builds nine blocks that put them back. A to G are cheap
per-document statistics; H and I are the two vectorizers.

**Everything is fitted on train only.** Blocks B-G are per-document functions and cannot
leak. Block A uses a pinned stop-word vocabulary and is not fitted at all. Only H and I
are fitted transforms, and section 3 asserts they saw no test text.

**What "did it improve" means here.** The supplied TF-IDF is re-run as a control in
section 5 of *this* notebook, on the same folds, in the same session. Comparing against
the stored `baseline_results.csv` from several weeks ago would confound the feature
change with anything else that has drifted since.

## 0. Setup

In [10]:
# Reload src/ helpers on every cell execution. Round 5 adds src/text.py,
# src/text_features.py and src/clustering.py while these notebooks are open, and a
# plain `import` caches the module in the kernel - so a fixed helper keeps failing
# with the OLD traceback until the kernel is restarted. With this, saving the .py
# is enough.
%load_ext autoreload
%autoreload 2

# Adds project root to path so `import src...` works from notebooks/.
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import paths, data, evaluation, tuning, ensemble, text
from src.paths import FIGURES
FIGURES.mkdir(parents=True, exist_ok=True)

from src import text_features as tf

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.ensemble import HistGradientBoostingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Load raw text and the locked split

Nothing re-splits. `dev_idx` is the same 16,000 rows every notebook since 01 has used,
so a score here is comparable to every score already recorded.

In [11]:
train_ids, train_texts, y = text.load_train_text()
test_ids, test_texts = text.load_test_text()

dev_idx = np.load(paths.DATA_PROCESSED / "dev_idx.npy")
holdout_idx = np.load(paths.DATA_PROCESSED / "holdout_idx.npy")
cv = evaluation.make_cv()

assert len(dev_idx) == 16000 and len(holdout_idx) == 4000
assert not set(dev_idx) & set(holdout_idx), "dev and holdout overlap"

y_dev = y[dev_idx]
print(f"train {len(train_texts)}, test {len(test_texts)}, dev {len(dev_idx)}")
print(f"dev machine share {y_dev.mean():.4f}")

train 20000, test 6999, dev 16000
dev machine share 0.6252


## 2. Sanity check: do the known signals reproduce?

The feature builder is new code operating on 27,000 documents, and a silent bug in it
would poison every result downstream. These differences were measured independently, by
a much simpler route, before the builder existed.

| statistic | human | machine |
|---|---|---|
| newlines per document | 7.70 | 4.47 |
| `**` occurrences per document | 0.049 | 0.496 |
| uppercase fraction | 0.0254 | 0.0230 |
| punctuation fraction | 0.0307 | 0.0266 |

In [12]:
summary = text.text_summary(train_texts, y)
print(summary.round(4).to_string())

expected = {"newlines_mean": (7.70, 4.47), "bold_mean": (0.049, 0.496),
            "upper_frac": (0.0254, 0.0230), "punct_frac": (0.0307, 0.0266)}
ok = True
for stat, (want0, want1) in expected.items():
    got0, got1 = summary.loc["label=0", stat], summary.loc["label=1", stat]
    close = (abs(got0 - want0) < 0.05 * want0 + 1e-4
             and abs(got1 - want1) < 0.05 * want1 + 1e-4)
    ok &= close
    print(f"  {stat:16s} got ({got0:.4f}, {got1:.4f})  "
          f"want ({want0}, {want1})  {'ok' if close else 'MISMATCH'}")

assert ok, "raw-text statistics do not match what was measured independently"
print("\nSanity check passed - the corpus is what the exploration described.")

             n  char_mean  char_median  word_mean  newlines_mean  bold_mean  upper_frac  punct_frac
group                                                                                              
label=0   7496  1529.4469       1065.0   261.4636         7.7036     0.0491      0.0254      0.0310
label=1  12504  1445.1233       1198.0   237.5543         4.4676     0.4961      0.0230      0.0266
  newlines_mean    got (7.7036, 4.4676)  want (7.7, 4.47)  ok
  bold_mean        got (0.0491, 0.4961)  want (0.049, 0.496)  ok
  upper_frac       got (0.0254, 0.0230)  want (0.0254, 0.023)  ok
  punct_frac       got (0.0310, 0.0266)  want (0.0307, 0.0266)  ok

Sanity check passed - the corpus is what the exploration described.


## 3. Build and cache the nine blocks

Cached to `data/processed/textfeat_<block>.npz`, so this runs once per machine. The
caches are gitignored like every other derived artifact and rebuilding is deterministic.

Blocks H and I are the expensive ones (two TF-IDF fits over 20,000 documents); A to G
are a single pass over each corpus.

In [13]:
REBUILD = False   # set True to force a rebuild after editing src/text_features.py

built = {}
for block in tf.ALL_BLOCKS:
    if not REBUILD and tf.cache_path(block).exists():
        built[block] = tf.load_block(block)
        print(f"{block:20s} cached")
        continue
    print(f"{block:20s} building ...", flush=True)
    one = tf.build_blocks(train_texts, test_texts, blocks=[block])[block]
    tf.save_block(block, one)
    built[block] = one
    print(f"{block:20s} built {one['train'].shape}")

A_function_words     cached
B_punctuation        cached
C_casing             cached
D_structure          cached
E_length             cached
F_diversity          cached
G_readability        cached
H_char_ngrams        cached
I_word_ngrams        cached


In [14]:
# Correctness guards. Each catches a specific failure mode that would otherwise surface
# only as a mediocre score weeks later.
inventory = []
for block, entry in built.items():
    Xtr, Xte = entry["train"], entry["test"]
    assert Xtr.shape[0] == 20000 and Xte.shape[0] == 6999, (block, Xtr.shape, Xte.shape)
    assert Xtr.shape[1] == Xte.shape[1] == len(entry["names"]), block
    assert np.isfinite(Xtr.data).all() and np.isfinite(Xte.data).all(), block
    inventory.append({"block": block, "n_features": Xtr.shape[1],
                      "density": Xtr.nnz / (Xtr.shape[0] * Xtr.shape[1]),
                      "mb": (Xtr.data.nbytes + Xtr.indices.nbytes) / 1e6})

print(pd.DataFrame(inventory).round(4).to_string(index=False))
print("\nEvery block: 20000 train rows, 6999 test rows, names aligned, all finite.")

           block  n_features  density       mb
A_function_words         318   0.1287   9.8241
   B_punctuation          37   0.1892   1.6797
        C_casing           5   0.5874   0.7048
     D_structure          10   0.4957   1.1896
        E_length          10   0.9939   2.3853
     F_diversity           5   0.9948   1.1938
   G_readability           5   0.9972   1.1966
   H_char_ngrams       20000   0.0746 238.6045
   I_word_ngrams       20000   0.0089  28.3895

Every block: 20000 train rows, 6999 test rows, names aligned, all finite.


In [15]:
# Leakage guard. H and I are the only fitted transforms. Refitting block I on the
# training text alone must reproduce the cached matrix exactly; had the cache been built
# on train+test, the vocabulary would differ and this would fail.
probe_tr, probe_te, probe_names = tf.word_ngram_features(train_texts, test_texts)
cached = built["I_word_ngrams"]

same_vocab = probe_names == cached["names"]
same_matrix = bool(abs(probe_tr - cached["train"]).max() < 1e-6) if same_vocab else False
print(f"block I vocabulary identical on refit:   {same_vocab}")
print(f"block I train matrix identical on refit: {same_matrix}")
assert same_vocab and same_matrix, \
    "block I is not reproducible from train alone - it may have been fitted on test"
print("\nNo vectorizer has seen test text.")

block I vocabulary identical on refit:   True
block I train matrix identical on refit: True

No vectorizer has seen test text.


## 4. The representations to compare

| key | contents |
|---|---|
| `tfidf_supplied` | the course's 5,000 lemma features, **the control** |
| `block_<X>` | each new block alone, to see what each is worth |
| `style_all` | blocks A-G, the cheap non-vectorizer features |
| `text_all` | blocks A-I, everything built from raw text |
| `supplied_plus_style` | the course features plus A-G |
| `supplied_plus_all` | the course features plus everything |

The single-block rows are what make section 5 diagnostic rather than a horse race. If
`block_H` alone is close to `text_all`, the gain is one block and the other eight are
decoration, which is a cheaper and more honest thing to report.

In [16]:
from scipy import sparse as sp

X_supplied, _, _ = data.load_train_features(sparse=True)

STYLE = ["A_function_words", "B_punctuation", "C_casing", "D_structure",
         "E_length", "F_diversity", "G_readability"]


def rep(name, rows=None):
    '''Build one named representation, restricted to `rows` (default: dev).'''
    rows = dev_idx if rows is None else rows
    if name == "tfidf_supplied":
        return X_supplied[rows]
    if name.startswith("block_"):
        block = [b for b in tf.ALL_BLOCKS if b.startswith(name[6:] + "_")][0]
        return built[block]["train"][rows]
    if name == "style_all":
        return tf.stack(built, STYLE)[0][rows]
    if name == "text_all":
        return tf.stack(built, tf.ALL_BLOCKS)[0][rows]
    if name == "supplied_plus_style":
        return sp.hstack([X_supplied[rows], tf.stack(built, STYLE)[0][rows]],
                         format="csr")
    if name == "supplied_plus_all":
        return sp.hstack([X_supplied[rows], tf.stack(built, tf.ALL_BLOCKS)[0][rows]],
                         format="csr")
    raise KeyError(name)


REPRESENTATIONS = (["tfidf_supplied"]
                   + [f"block_{b[0]}" for b in tf.ALL_BLOCKS]
                   + ["style_all", "text_all", "supplied_plus_style",
                      "supplied_plus_all"])
for name in REPRESENTATIONS:
    print(f"{name:22s} {rep(name).shape}")

tfidf_supplied         (16000, 5000)
block_A                (16000, 318)
block_B                (16000, 37)
block_C                (16000, 5)
block_D                (16000, 10)
block_E                (16000, 10)
block_F                (16000, 5)
block_G                (16000, 5)
block_H                (16000, 20000)
block_I                (16000, 20000)
style_all              (16000, 390)
text_all               (16000, 40390)
supplied_plus_style    (16000, 5390)
supplied_plus_all      (16000, 45390)


## 5. The baseline suite

Five to six models spanning the families in `baseline_results.csv`, under the locked
5-fold stratified CV on Macro F1. **Default hyperparameters throughout**: this measures
the representation, and tuning each model per representation would confound the two and
cost days.

`tuning.run_trial` caches every result by a hash of its configuration, so this is
resumable and merges across teammates through `data/processed/tuning_trials/` exactly as
the round-4 searches did. No new ledger.

**Change `ME` and nothing else.** Split by cost: blocks A-G are small and fast, H and I
are 20,000-column sparse matrices and much slower.

In [17]:
ASSIGNMENTS = {
    "cliffton": ["tfidf_supplied", "block_A"],
    "brian":    [f"block_{b[0]}" for b in tf.DENSE_BLOCKS],
    "jovyan":   ["block_H", "block_I", "text_all"],
    "koko":     ["style_all", "supplied_plus_style", "supplied_plus_all"],
}

ME = "koko"   # <-- THE ONLY LINE TO CHANGE

n_pos, n_neg = int((y_dev == 1).sum()), int((y_dev == 0).sum())

MODELS = {
    "lightgbm":     lambda: LGBMClassifier(class_weight="balanced", verbose=-1,
                                           n_jobs=-1, random_state=42),
    "xgboost":      lambda: XGBClassifier(tree_method="hist", eval_metric="logloss",
                                          scale_pos_weight=n_neg / n_pos, n_jobs=-1,
                                          random_state=42),
    "logreg":       lambda: LogisticRegression(max_iter=2000, class_weight="balanced",
                                               random_state=42),
    "linearsvc":    lambda: LinearSVC(class_weight="balanced", max_iter=5000,
                                      dual="auto", random_state=42),
    "complementnb": lambda: ComplementNB(),
}

# HistGB needs dense input, so it only runs where densifying is affordable. Skipping it
# elsewhere is printed rather than silent.
DENSE_ONLY = {"histgb": lambda: HistGradientBoostingClassifier(random_state=42)}
DENSE_LIMIT = 400

print(f"{ME} runs: {ASSIGNMENTS[ME]}")

koko runs: ['style_all', 'supplied_plus_style', 'supplied_plus_all']


In [18]:
for rep_name in ASSIGNMENTS[ME]:
    X = rep(rep_name)
    print(f"\n=== {rep_name}  {X.shape}", flush=True)

    todo = dict(MODELS)
    if X.shape[1] <= DENSE_LIMIT:
        todo.update(DENSE_ONLY)
    else:
        print(f"  (histgb skipped: {X.shape[1]} columns is too many to densify)")

    for clf_name, make in todo.items():
        Xf = X.toarray() if clf_name == "histgb" else X
        # ComplementNB requires non-negative input. Block E carries log counts and
        # block G carries Flesch scores, both of which go negative.
        if clf_name == "complementnb" and Xf.min() < 0:
            print(f"  {clf_name:14s} skipped (representation has negative values)")
            continue
        params = {"clf": clf_name, "representation": rep_name}
        record = tuning.run_trial(f"repr_{rep_name}", ME, make(), params, Xf, y_dev, cv)
        print(f"  {clf_name:14s} {record['mean']:.4f} +/- {record['std']:.4f}")


=== style_all  (16000, 390)


C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid featur

  lightgbm       0.8452 +/- 0.0056
  xgboost        0.8458 +/- 0.0080


C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/

  logreg         0.6903 +/- 0.0034
  linearsvc      0.7369 +/- 0.0069
  complementnb   skipped (representation has negative values)
  histgb         0.8435 +/- 0.0034

=== supplied_plus_style  (16000, 5390)
  (histgb skipped: 5390 columns is too many to densify)


...

C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/

  logreg         0.7030 +/- 0.0069


C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\svm\_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\svm\_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\svm\_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\svm\_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


  linearsvc      0.5332 +/- 0.1408
  complementnb   skipped (representation has negative values)


C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\svm\_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


### Commit your trials

```bash
git add data/processed/tuning_trials/
git commit -m "feat: representation baselines for <your representations>"
git push
```

Section 6 needs everyone's, so `git pull` first.

## 6. The merged comparison

The control matters more than the winner. `tfidf_supplied` re-run here must land close to
`baseline_results.csv` (LightGBM 0.73914, LinearSVC 0.72278, ComplementNB 0.65798). If it
lands far *above* those, something leaked and no other row can be trusted.

In [ ]:
frames = []
for rep_name in REPRESENTATIONS:
    trials = tuning.load_trials(f"repr_{rep_name}")
    for _, row in trials.iterrows():
        frames.append({"representation": rep_name, "clf": row["params"]["clf"],
                       "owner": row["owner"], "mean": row["mean"], "std": row["std"]})

results = pd.DataFrame(frames)
assert len(results), "no representation trials on disk - has anyone run section 5?"

missing = [r for r in REPRESENTATIONS if r not in set(results["representation"])]
if missing:
    print(f"still waiting on: {missing}\n")

table = (results.pivot_table(index="representation", columns="clf", values="mean")
         .reindex([r for r in REPRESENTATIONS
                   if r in set(results["representation"])]))
table["best"] = table.max(axis=1)
print(table.round(4).to_string())

STORED = {"lightgbm": 0.73914, "linearsvc": 0.72278, "complementnb": 0.65798}
if "tfidf_supplied" in table.index:
    print("\nControl check against baseline_results.csv:")
    for clf, stored in STORED.items():
        if clf in table.columns and pd.notna(table.loc["tfidf_supplied", clf]):
            got = float(table.loc["tfidf_supplied", clf])
            flag = "LEAK?" if got > stored + 0.02 else "ok"
            print(f"  {clf:14s} now {got:.4f}  stored {stored:.4f}  "
                  f"delta {got - stored:+.4f}  {flag}")

In [ ]:
control = float(table.loc["tfidf_supplied", "best"])
gains = (table["best"] - control).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
colors = ["tab:gray" if i == "tfidf_supplied"
          else ("tab:green" if v > 0 else "tab:red") for i, v in gains.items()]
ax.barh(range(len(gains)), gains.to_numpy(), color=colors)
ax.set_yticks(range(len(gains)), gains.index, fontsize=8)
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel(f"best-model Macro F1 minus the supplied-TF-IDF control ({control:.4f})")
ax.set_title("Representation comparison, locked 5-fold CV", fontsize=10)
ax.grid(alpha=0.3, axis="x")
plt.tight_layout()
plt.savefig(FIGURES / "representation_comparison.png", dpi=120)
plt.show()

results.to_csv(paths.DATA_PROCESSED / "representation_results.csv", index=False)
print(f"control (supplied TF-IDF, best model): {control:.4f}\n")
print(gains.round(4).to_string())
print("\nCV std is roughly 0.004, so treat anything under about 0.008 as a tie.")

## Discussion / carry-forward -> `13_clustering.ipynb`

_Fill in once every representation has been run and merged._

**Record all four of these, whatever they say:**

1. **Did the control reproduce?** If `tfidf_supplied` differs from `baseline_results.csv`
   by more than CV noise, find out why before reading anything else in the table.
2. **Which single block carries the most?** If `block_H` is close to `text_all`, the gain
   is one block and the rest are decoration.
3. **Does `block_I` beat `tfidf_supplied`?** Both are word TF-IDF over the same corpus;
   the only differences are that block I keeps stop words and does not lemmatize. A win
   localises the loss to the course preprocessing, which is a clean reportable finding.
4. **Do the combinations beat their parts?** If `supplied_plus_style` beats both
   `tfidf_supplied` and `style_all`, content and style are complementary.

**The caveat that governs how much any of this is worth.** These numbers come from
standard 5-fold CV, which trains and tests on the same domains. The test set is a
different domain, so a representation can win this table by memorising training-domain
vocabulary and transfer nothing. That is exactly the failure mode behind three rounds of
non-transferring gains.

**Do not ship a feature set on the strength of this table alone.** Notebook 13 builds the
domain clusters and notebook 14 re-scores every block under leave-one-cluster-out CV. The
gap between the two protocols is what says which of these gains are real.